In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import mplleaflet
import numpy as np
import time
import csv
import rospy
import yaml

from sensor_msgs.msg import NavSatFix

kml_file_path = os.getenv('RUN_LIVE_VIEW_YAML_FILE_PATH')
gpd.io.file.fiona.drvsupport.supported_drivers['KML'] = 'rw'
df = gpd.read_file(kml_file_path, driver='KML')
ax = df.plot()

longs = []
lats = []
i = 0

def callback(msg):
    global ax, df, lats, longs, i
    
    if i>=50:
        i = 0
        lats.append(msg.latitude)
        longs.append(msg.longitude)
    else:
        i += 1
        
mplleaflet.save_html(fig=ax.figure)
rospy.init_node('map_vis_live')
rospy.Subscriber('/agribot/gnss', NavSatFix, callback)
rate = rospy.Rate(0.2)

while not rospy.is_shutdown():
    if len(lats)>0:
        ax.plot(longs, lats, 'r.')
        mplleaflet.save_html(fig=ax.figure)
        lats = []
        longs = []
    rate.sleep()